# MySQL y MongoDB: seis operaciones completas

Los CSV y `results.json` contienen la última medición realizada con `benchmark.py`.
Las unidades son **segundos**. El gráfico muestra mediana y rango, tras un calentamiento.
Para repetir la prueba en Docker, véase `README.md`.

La prueba utiliza los mismos datos, rangos y resultados. MySQL confirma cada lote en una transacción;
MongoDB confirma escrituras con diario y mantiene atomicidad por documento. No mide escalabilidad
y no iguala las garantías transaccionales. Los ficheros `output.*` son registros históricos de 2016.


In [ ]:
from pathlib import Path
from typing import Any
import json
import pandas as pd
import matplotlib.pyplot as plt

root: Path = Path.cwd()
report: dict[str, Any] = json.loads((root / "results.json").read_text())
assert report["unit"] == "s"
mysql: pd.DataFrame = pd.read_csv(root / "mysql.csv", index_col="query")
mongo: pd.DataFrame = pd.read_csv(root / "mongo.csv", index_col="query")
assert mysql.index.equals(mongo.index)
print(report["measured_at_utc"], report["versions"], report["rows"], "registros")
comparison: pd.DataFrame = pd.DataFrame({"MySQL (s)": mysql.duration_s, "MongoDB (s)": mongo.duration_s})
comparison


In [ ]:
figure: Any
axis: Any
figure, axis = plt.subplots(figsize=(12, 5.1), layout="constrained")
positions: list[int] = list(range(len(mysql)))
for engine, data, shift, color in (("MySQL", mysql, -0.19, "#c98b00"), ("MongoDB", mongo, 0.19, "#a9251a")):
    errors: list[list[float]] = [(data.duration_s - data.min_s).tolist(), (data.max_s - data.duration_s).tolist()]
    axis.bar([position + shift for position in positions], data.duration_s, width=0.36,
             color=color, label=engine, yerr=errors, capsize=4)
axis.set_xticks(positions, ["Carga", "Índice", "Actualización", "Borrado", "Recuento", "Intervalo"])
axis.set_yscale("log")
axis.set_ylabel("Tiempo completo (s), escala logarítmica")
axis.legend()
axis.grid(axis="y", alpha=0.25)
axis.set_axisbelow(True)
axis.set_title(f'{report["rows"]:,} registros · mediana y rango de {report["repeats"]} repeticiones')
plt.show()


## Interpretación

- ¿Qué parte del tiempo de carga corresponde al cliente, al envío y a la confirmación?
- ¿Por qué no equivale contar metadatos a calcular el recuento exacto?
- ¿Cómo cambiaría la consulta si sólo construyéramos un cursor sin recorrerlo?
- ¿Qué puede cambiar al añadir concurrencia, otro índice o varias réplicas?
- En escala logarítmica, distancias iguales representan factores iguales
